In [10]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
class Performer:
    full_history = [] #class variable: full history of the game
    
    def __init__(self, name: str, role: str, client: any, model: str, system_prompt: str="", 
                 temperature: float=0.7, max_tokens: int=500):
        """
        A simple performer in the Yes-And game.

        Parameters:
        - name: The name of the performer (e.g., "Drew", "Wayne", "Ryan")
        - role: "host" or "player"
        - client: the AI client object (e.g., OpenAI(), Anthropic(), etc.)
        - model: the model string to use (e.g., "gpt-4.1-mini")
        - system_prompt: the system prompt for this performer
        - temperature: randomness (default 0.7)
        - max_tokens: max tokens in the response (default 500)
        """
        self.name = name
        self.role = role
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.temperature = temperature
        self.max_tokens = max_tokens
    
    def set_system_prompt(self, system_prompt: str):
        """Reset or update the system prompt."""
        self.system_prompt = system_prompt
    
    def start_game(self, user_message: str):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        print(response)
        return response.choices[0].message.content

    def performer_user_interaction(self, user_message: str):
        """Call the performer (LLM client) with the system prompt + user message."""
        messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_message}
            ],
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        return response.choices[0].message.content

In [57]:
load_dotenv(override=True)

# create clients for different models
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI()
anthropic = OpenAI(api_key=os.getenv('ANTHROPIC_API_KEY'), base_url=anthropic_url)
gemini = OpenAI(api_key=os.getenv('GOOGLE_API_KEY'), base_url=gemini_url)
# instantiate performers
host = Performer(name="Drew", role="host", client=openai, model="gpt-4o-mini")
performer1 = Performer(name="Ryan", role="player", client=gemini, model="gemini-2.5-flash")
performer2 = Performer(name="Wayne", role="player", client=anthropic, model="claude-3-5-haiku-latest")

In [ ]:

host_system_prompt = f"""You are {host.name}, the host of a multi-agent “Yes, And” improv game.You do not play the game yourself—you only guide it, 
moderate it, and make decisions.
Responsibilities
Solicit Scenario
Ask the user (audience) for a fun scenario to start the game.If unclear/inappropriate, ask them to rephrase once.
Frame the Scene
Convert the scenario into a structured Scene Brief (setting, characters, tone, and constraints).
Broadcast this Scene Brief as instructions to the players.
Run the Game Loop
Alternate turns between {performer1.name} (Player 1) and {performer2.name} (Player 2).
After each pair of turns, decide whether to continue or end.
If ending, wrap up with a closing message to the user.
Decision Making
Say [HOST DECISION: continue] to keep the game going.
Say [HOST DECISION: End Game] to stop."""

performer1_system_prompt = f"""You are {performer1.name}, Player 1 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer2.name} — only roleplay your own turn."""

performer2_system_prompt = f"""You are {performer2.name}, Player 2 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer1.name} — only roleplay your own turn."""

In [58]:
host.set_system_prompt(host_system_prompt)
response = host.start_game("Please tell me what you need to get the game started?")
print(response)

ChatCompletion(id='chatcmpl-CN7OucZJOTIdFyEx6t5P2lIgeALeW', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I just need a fun scenario from you to kick off the game! It could be anything—an unusual situation, a setting, or a concept. Please share your idea, and I'll turn it into a structured scene for our players!", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1759626028, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_560af6e559', usage=CompletionUsage(completion_tokens=47, prompt_tokens=202, total_tokens=249, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
I just need a fun scenario from you to kick off the game! It could be anything—an unus